# 02 · Prepare and train the research baseline

Run this notebook only after installing `ml-service` and placing the original raw `.psv` training files under `ml-service/data/raw/`. It calls the same source-controlled CLI scripts that are used outside Jupyter.

**Safety:** This is a first software/ML baseline, not a clinically validated model. The target is inferred sepsis onset in the next six hourly positions, excluding post-onset rows. The threshold of 0.5 is a technical benchmark, not a clinical decision cutoff. No model score should modify the ICU queue or allocate a bed.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

CWD = Path.cwd().resolve()
ML_ROOT = CWD if (CWD / 'data' / 'raw').exists() else (CWD.parent.parent / 'ml-service' if CWD.name == 'ml' else CWD / 'ml-service')
if not (ML_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError(f'Cannot locate ml-service/ from {CWD}')
print('ML service:', ML_ROOT)
print('Python:', sys.executable)

In [ ]:
prepare_command = [sys.executable, '-m', 'bedallocation_ml.prepare_dataset', '--raw-dir', 'data/raw', '--output-dir', 'data/processed_onset']
subprocess.run(prepare_command, cwd=ML_ROOT, check=True)

In [ ]:
train_command = [sys.executable, '-m', 'bedallocation_ml.train_baseline', '--data-dir', 'data/processed_onset', '--artifact-dir', 'artifacts']
subprocess.run(train_command, cwd=ML_ROOT, check=True)

In [ ]:
metrics_path = ML_ROOT / 'artifacts' / 'sepsis_onset_next_6h_evaluation_metrics.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print('Target:', metrics['target_definition'])
print('Clinical status:', metrics['clinical_status'])
display(pd.DataFrame(metrics['metrics']).T)

## How to read the metrics

- **Average precision / PR-AUC:** useful for understanding precision-recall performance when positives are uncommon.
- **ROC-AUC:** ranking quality across thresholds; it does not guarantee clinical usefulness.
- **Precision and recall at 0.5:** technical reference metrics only; class weighting means returned scores are not automatically calibrated probabilities.
- **Patient-level split:** helps prevent overlapping hourly rows from the same patient appearing in train and test.

Keep the test results untouched during model development. Before any clinical use, the target definition, data representativeness, external validation, calibration, workflow, and risks require independent expert review. This model is not an ICU admission predictor.